# guidance

In [2]:
%load_ext autoreload

%autoreload 2

In [3]:
cd ..

/Users/alain/Desktop/master-thesis/guiding-diffusion-based-weather-models


In [20]:
from src import guidance_funcs, rollout, guidance_funcs, helpers, sampler

In [ ]:
import torch

# Example shapes
B, C, H, W = 1, 4, 16, 16
device = "cpu"

SIGMA = torch.ones(1, C, 1, 1, device=device)

def deterministic(x_cond):
    # Example: persistence-like mean
    return 0.5 * (x_cond[0] + x_cond[1])

def denoiser(z, x_cond, t):
    # Dummy denoiser for demonstration
    return z

x0 = torch.randn(B, C, H, W, device=device)
x1 = torch.randn(B, C, H, W, device=device)
x_cond = [x0, x1]

mask = torch.zeros(1, 1, H, W, device=device)
mask[..., 4:8, 4:8] = 1.0

# Scalar target for masked mean
x_guide_trajectory = [torch.tensor(1.5, device=device) for _ in range(3)]

# define inside diffusion object
sample_fn = sampler.make_sampler(
    sigma=SIGMA,
    denoiser=denoiser,
    decode_in_sampler=True,
)

guide_fn = guidance_funcs.make_no_guidance()

traj = rollout.rollout(
    N=3,
    T=10,
    x_cond=x_cond,
    # TODO: maybe collapse these two together
    deterministic=deterministic,
    sample=sample_fn,
    guide=guide_fn,
    x_guide_trajectory=x_guide_trajectory,
    mask=mask,
)

print(len(traj), traj[0].shape)

TypeError: 'ellipsis' object is not callable

### reference rollout routine

In [ ]:
def sample_rollout(
    self,
    batch,
    batch_nb=0,
    member=0,
    iterations=1,
    disable_tqdm=False,
    return_format="tensordict",
    **kwargs,
):
    torch.set_grad_enabled(False)
    preds_future = []
    loop_batch = {k: v for k, v in batch.items()}

    for i in tqdm(range(iterations), disable=disable_tqdm):
        seed_i = member + 1000 * i + batch_nb * 10**6
        sample = self.sample(loop_batch, seed=seed_i, disable_tqdm=True, **kwargs)
        preds_future.append(sample)
        loop_batch = dict(
            prev_state=loop_batch["state"],
            state=sample,
            timestamp=loop_batch["timestamp"] + batch["lead_time_hours"] * 3600,
        )

    if return_format == "list":
        return preds_future
    preds_future = torch.stack(preds_future, dim=1)
    return preds_future
